# 7 (Capstone) - Composing the 120-Scenario Suite

Chapter 7 composes a base and its factors into a concrete suite. This companion runs that composition on the capstone: the **twenty seed cases** of Chapter 5 and the **three presentation factors** of Chapter 6 become a fixed **120-scenario** campaign.

Two design choices make the campaign read cleanly. The suite is **embedded** --- the seed case is a factor of the design, not crossed against it --- so the suite is a fixed size and a failure can be attributed to the case itself. And because a 20-level base would be starved by a joint space-filling design at this budget, the seed case enters as a **balanced blocking factor**: every case is exercised equally. The section *Where the 120 comes from* below shows exactly how the budget, the Sobol design and the blocking combine --- they are two independent steps, not one.

## Compose the suite

We use the same Ch16 parity config the campaign uses — the three factor names and the domain level vocabulary (`level_overrides`) that renames the catalog's generic levels into the banking terms of Chapter 6 (inlined as `pc` in the next cell). The base items come from Chapter 5's `SeedCaseSource`; `gt.compose` pairs them with the space-filled factor design.

**What the next cell does** — composes the 120-scenario suite from the catalog, the seed cases and the application config:

1. **Locate root.** Walks up from the cwd to find `ROOT` (the repo root holding `data/eval_cases/cases.json`), then imports `proofloop.suite as gt` and `graphdoe_design`.
2. **Load inputs.** Loads the factor catalog with `gt.Catalog.load()`, the twenty seed cases via `gt.SeedCaseSource(...).items()`, and defines the `parity_ch16` config inline as `pc` (its `factors` and `level_overrides`).
3. **Resolve the suite.** Calls `gt.resolve(cat, ['conditional_rule'], pc['factors'], mode='embedded')` to bind the base and the three presentation factors.
4. **Compose.** Calls `gt.compose(suite, items, n_runs=120, seed=42, design_fn=partial(graphdoe_design, method='sobol+refine'), balance_base=True, level_overrides=pc['level_overrides'])`, producing `scns` — the 120 Sobol-spread, case-balanced scenarios — and prints the count, factor names and a sample row.

In [ ]:
import pathlib, json
from functools import partial
from collections import Counter

import proofloop.suite as gt
from proofloop.suite.compose import graphdoe_design

# labeled seed cases bundled in this repo's data/
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / 'data' / 'eval_cases' / 'cases.json').exists())

# Ch16 parity config (presentation factors + level vocabulary for the complaint agent)
pc = {
    'factors': ['clarity', 'entity_aliasing', 'reasoning_cue'],
    'mode': 'embedded',
    'balance_base': False,
    'design_method': 'sobol',
    'level_overrides': {
        'clarity':         ['clear', 'ambiguous', 'misleading'],
        'entity_aliasing': ['canonical', 'alias', 'typo'],
        'reasoning_cue':   ['none', 'cot', 'misleading_cue'],
    },
}

cat = gt.Catalog.load()
items = gt.SeedCaseSource(json.loads((ROOT / 'data' / 'eval_cases' / 'cases.json').read_text())).items()

suite = gt.resolve(cat, ['conditional_rule'], pc['factors'], mode='embedded')
scns = gt.compose(suite, items, n_runs=120, seed=42,
                  design_fn=partial(graphdoe_design, method='sobol+refine'),
                  balance_base=True, level_overrides=pc['level_overrides'])

print(f'{len(scns)} scenarios from {len(items)} cases x {len(suite.factor_names)} factors')
print('factors      :', suite.factor_names)
print('sample row   :', scns[0].base.qid, '|', scns[0].factor_levels)

## Where the 120 comes from

120 is not computed from the factor math --- it is the **budget** you choose (`n_runs`), the number of agent calls you can afford. The full grid would be `3 x 3 x 3 = 27` presentation combinations times 20 cases `= 540`; the design covers that space with only 120 runs instead of all 540.

Then, under `balance_base=True`, the 120 are produced by **two independent steps that are zipped together** --- *not* a per-case Sobol:

1. **Sobol space-fills the three presentation factors, once, for all 120 rows.** The seed case is not in this design. This is one 120-point space-filling sample in the 3-factor space; it is where the 47/27/46 spread comes from.
2. **The 20 cases are assigned separately, as a balanced shuffle** --- the list `[case-001 ... case-020] x 6`, shuffled, so each case appears exactly six times.
3. **Zip them row by row.** Row i = (Sobol presentation row i, assigned case i).

So a case's six rows are not a designed per-case sample --- they are just whichever six of the global 120 Sobol rows happened to receive that case's label. The cell below reproduces the two steps explicitly.

**What the next cell does** — reproduces the two independent steps `compose` runs internally, making the 120 explicit:

1. **Import the internal helper.** Imports `_balanced_assignment` from `proofloop.suite.compose`, the exact function `compose` uses to assign cases.
2. **Step 1 — Sobol design.** Builds `pres_specs` from `pc['factors']` and their `level_overrides`, then calls `graphdoe_design(pres_specs, n_runs=120, seed=42)` to space-fill the three presentation factors only — the case is not a dimension here, yielding `pres_rows`.
3. **Step 2 — case overlay.** Calls `_balanced_assignment([it.qid for it in items], 120, seed=42)` to get `assign`, a balanced shuffle of the 20 case ids, six each.
4. **Step 3 — zip.** Confirms row i pairs Sobol presentation row i with assigned case i, which is exactly the 120 scenarios from the first cell.

In [ ]:
from proofloop.suite.compose import _balanced_assignment   # the same step compose runs internally

# the full grid we are NOT running
print(f'full grid would be 3 x 3 x 3 x 20 cases = {3**3 * len(items)} scenarios; budget n_runs = 120\n')

# Step 1 -- Sobol over the THREE presentation factors only, once, at size 120
pres_specs = [(f, pc['level_overrides'][f]) for f in pc['factors']]
pres_rows = graphdoe_design(pres_specs, n_runs=120, seed=42)
print(f'step 1  Sobol design : {len(pres_rows)} rows over {len(pres_specs)} factors '
      f'(the case is NOT a dimension here)')

# Step 2 -- assign the 20 cases as a separate balanced shuffle, 6 each
assign = _balanced_assignment([it.qid for it in items], 120, seed=42)
print(f'step 2  case overlay : {len(assign)} labels, '
      f'per-case counts {sorted(set(Counter(assign).values()))} (six each, independent of Sobol)')

# Step 3 -- zip: row i = (presentation row i, case i). This IS the suite from the cell above.
print('step 3  zip          : 120 (presentation row, case) pairs == the 120 scenarios above')

## Balanced blocking: every case exercised equally

Because the seed case is a blocking factor, the 120 scenarios split evenly over the 20 cases --- exactly six each. That equal allocation is what lets the factor attribution of Chapter 10 treat the seed case as a controlled block rather than a confound. The crossed arrangement, by contrast, would pair every case with every design row and grow with the number of cases.

**What the next cell does** — verifies the seed-case balance and contrasts it with the crossed arrangement:

1. **Count per case.** Builds `seed_counts` as a `Counter` of `s.base.qid` over `scns`, then prints the distinct-case count and the set of per-case counts (six each).
2. **Build the crossed contrast.** Calls `gt.compose` on a `gt.resolve(..., mode='cross')` suite over a single factor with `n_runs=8`, producing `cross`.
3. **Show the growth.** Prints `len(cross)` (20 cases x 8 runs = 160), demonstrating that the crossed arrangement scales with the number of base cases while the embedded suite stays fixed at 120.

In [ ]:
seed_counts = Counter(s.base.qid for s in scns)
print('embedded suite : 120 scenarios, seed case balanced')
print('  distinct cases :', len(seed_counts))
print('  per-case counts:', sorted(set(seed_counts.values())), '(six each)')

# contrast: the crossed arrangement grows with the number of cases
cross = gt.compose(gt.resolve(cat, ['conditional_rule'], ['clarity'], mode='cross'),
                   items, n_runs=8, seed=42,
                   design_fn=partial(graphdoe_design, method='sobol+refine'))
print(f'\ncrossed (20 cases x 8 runs) : {len(cross)} scenarios  (grows with the base)')

## Space-filling: the presentation factors are spread, not balanced

The blocking holds the *base* balanced; the *presentation factors* are space-filled by Sobol+refine, so their marginals are near-uniform but not forced equal. The small departures from a perfect 40/40/40 are the cost of filling the joint space evenly rather than balancing each factor in isolation --- and they are exactly reproducible from the seed.

**What the next cell does** — prints the marginal level counts for each presentation factor:

1. **Loop the factors.** Iterates `suite.factor_names` (`clarity`, `entity_aliasing`, `reasoning_cue`).
2. **Tally each marginal.** For each factor, builds a `Counter` of `s.factor_levels[f]` over the 120 scenarios, showing the near-uniform but not exactly equal spread that Sobol+refine produces.

In [ ]:
for f in suite.factor_names:
    print(f'{f:16s}:', dict(Counter(s.factor_levels[f] for s in scns)))

## The same suite the campaign ran

This composition is not illustrative --- it is the campaign's. The pinned `capstone_run.json` records the balance of the run that produced every number in Chapter 12, and the live design above reproduces it from the seed: the same six-each blocking and the same clarity spread. Chapter 8 then materializes each row into an actual customer message (notebook 12.3), and Chapter 10 attributes the outcome to these factors (notebook 12.6).

**What the next cell does** — checks the live design against the pinned campaign artifact:

1. **Load the pinned run.** Reads `data/capstone_run.json` into `pinned`.
2. **Recompute the live spread.** Builds `live_clarity` as a `Counter` of `s.factor_levels['clarity']` over the 120 scenarios.
3. **Compare.** Prints `pinned['clarity_balance']` against `live_clarity` and the boolean `live_clarity == pinned['clarity_balance']`, then confirms `pinned['seed_balance']` is six per case — the live seed reproduces the campaign's design exactly.

In [ ]:
pinned = json.loads((ROOT / 'data' / 'capstone_run.json').read_text())   # pinned GMS Sobol run
live_clarity = dict(Counter(s.factor_levels['clarity'] for s in scns))
match = live_clarity == pinned['clarity_balance']
print('pinned clarity_balance :', pinned['clarity_balance'], '(GMS Sobol+refine run)')
print('live  clarity spread   :', live_clarity)
print('match                  :', match,
      '' if match else '(exact parity needs the licensed GMS Sobol backend; open uses simple_design)')
print('pinned seed balance    :', sorted(set(pinned['seed_balance'].values())), '(six each)')

## Self-check

The asserts below pin every number this notebook claims --- a 120-scenario embedded suite, six scenarios per case, a crossed arrangement that grows with the base, and a clarity spread that reproduces the pinned campaign artifact exactly. Run the notebook end to end and it fails loud if any of these drift, so the illustration stays reproducible rather than quietly going stale.

**What the next cell does** — asserts every claim in the notebook so it fails loud if the design drifts:

1. **Suite shape.** Asserts `len(scns) == 120` and `suite.mode == 'embedded'`.
2. **Balanced blocking.** Asserts the 120 cover exactly 20 distinct `seed_counts` keys with `{6}` as the only per-case count.
3. **Crossed contrast.** Asserts `len(cross) == 20 * 8`, confirming the crossed arrangement grows with the base.
4. **Factor identity and reproducibility.** Asserts `suite.factor_names` is the three presentation factors and `live_clarity == pinned['clarity_balance']`, reproducing the campaign artifact.
5. **Two-step decomposition.** Asserts `len(pres_rows) == 120` and the case overlay `Counter(assign)` is six each, pinning the Sobol-then-overlay construction.

In [ ]:
import forgeloop.gms as gms
assert len(scns) == 120 and suite.mode == 'embedded'
assert len(seed_counts) == 20 and set(seed_counts.values()) == {6}   # balanced blocking, six each
assert len(cross) == 20 * 8                                          # crossed grows with the base
assert set(suite.factor_names) == {'clarity', 'entity_aliasing', 'reasoning_cue'}
assert set(pinned['seed_balance'].values()) == {6}
assert len(pres_rows) == 120 and set(Counter(assign).values()) == {6}
if gms.available():
    assert live_clarity == pinned['clarity_balance']                # GMS Sobol reproduces the campaign design
    print('OK - Ch7 capstone: reproduces the licensed GMS campaign design')
else:
    print('OK - Ch7 capstone: 120 balanced-blocked, space-filled scenarios '
          '(open simple_design; exact Sobol parity is a GMS feature)')